# 02 · LoRA fine-tune a vision-language model on engineering drawings

Fine-tunes **Qwen2.5-VL-3B-Instruct** with LoRA (PEFT) on **MechVQA** (Apache-2.0, 3.3K drawings / 21K QA pairs) so the
extraction fallback reads mechanical drawings better. MechVQA's authors report a domain-tuned model beating the strongest
closed baseline by 7.57 points (see docs/RESEARCH.md, CLM-DRAW-2).

**Runtime: GPU (L4 or A100).** Honest note: LoRA for vision-language models on TPU via PyTorch/XLA is not well supported
in Colab today; use a GPU runtime for this notebook (QLoRA 4-bit fits a 3B VLM on a 24 GB L4). Run notebook 01 on TPU.

Output: a LoRA adapter you serve with vLLM (`--enable-lora`), which exposes an OpenAI-compatible API the backend already
speaks (`AERONEXUS_LLM_PROVIDER=openai`).

In [ ]:
# Get the repo, regenerate the synthetic corpus (deterministic, ~5 s) and fetch the public datasets (checksummed)
!git clone -q https://github.com/<you>/aeronexus-ai.git /content/aeronexus-ai || true
%cd /content/aeronexus-ai/backend
!pip -q install -r requirements.txt
!python scripts/generate_data.py --train 4000
!python scripts/fetch_public_data.py

In [ ]:
!pip -q install -U transformers peft trl datasets accelerate bitsandbytes qwen-vl-utils
import torch; print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — switch runtime")

In [ ]:
from datasets import load_dataset
raw = load_dataset("XiaofengAlg/MechVQA")
print(raw)                                  # inspect splits/columns — the mapping below adapts to common layouts
split_tr = "train" if "train" in raw else list(raw)[0]
cols = raw[split_tr].column_names; print(cols)

In [ ]:
from PIL import Image
import io, json

def pick(ex, *names):
    for n in names:
        if n in ex and ex[n] is not None:
            return ex[n]
    raise KeyError(f"none of {names} in {list(ex)}")

def to_chat(ex):
    img = pick(ex, "image", "images", "img")
    if isinstance(img, list): img = img[0]
    if isinstance(img, dict) and "bytes" in img: img = Image.open(io.BytesIO(img["bytes"]))
    if isinstance(img, str): img = Image.open(img)
    q = pick(ex, "question", "query", "prompt")
    a = pick(ex, "answer", "response", "label", "gt_answer")
    if isinstance(a, list): a = a[0]
    return {"messages": [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": str(q)}]},
        {"role": "assistant", "content": [{"type": "text", "text": str(a)}]}], "images": [img.convert("RGB")]}

train = raw[split_tr].shuffle(seed=0).select(range(min(6000, len(raw[split_tr])))).map(to_chat, remove_columns=cols)
evals = (raw["validation"] if "validation" in raw else raw[split_tr].select(range(200))).map(to_chat, remove_columns=cols)
print(train[0]["messages"])

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig
BASE = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(BASE, min_pixels=256*28*28, max_pixels=1024*28*28)  # cap tokens per drawing
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16))
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

In [ ]:
from trl import SFTConfig, SFTTrainer
cfg = SFTConfig(output_dir="vlm-lora", num_train_epochs=1, per_device_train_batch_size=2, gradient_accumulation_steps=8,
                learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.03, bf16=True, logging_steps=20,
                save_strategy="no", gradient_checkpointing=True, report_to=[], max_length=2048)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train, eval_dataset=evals, peft_config=lora,
                     processing_class=processor)
trainer.train()

In [ ]:
# Quick before/after check on held-out items: exact match after normalisation (+ judge later in the repo harness)
import re
def norm(s): return re.sub(r"[^a-z0-9.]", "", str(s).lower())
def answer(ex):
    msgs = ex["messages"][:1]
    text = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inp = processor(text=[text], images=ex["images"], return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=64, do_sample=False)
    return processor.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
sample = evals.select(range(min(100, len(evals))))
em = sum(norm(answer(e)) == norm(e["messages"][1]["content"][0]["text"]) for e in sample) / len(sample)
print("held-out exact match:", em)

In [ ]:
out = "/content/export/vlm-lora"
trainer.model.save_pretrained(out); processor.save_pretrained(out)
json.dump({"base_model": BASE, "dataset": "MechVQA (Apache-2.0)", "held_out_exact_match": em, "lora": lora.to_dict()},
          open(f"{out}/model_card.json", "w"), indent=1, default=str)
ARTIFACTS = ["/content/export/vlm-lora"]

In [ ]:
# Save artifacts to Google Drive (survives the Colab session; avoids flaky browser downloads of big files).
# In VS Code: python scripts/import_models.py <the downloaded aeronexus_export.zip or Drive folder>
import hashlib, json, os, platform, shutil, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/aeronexus/export"
os.makedirs(OUT, exist_ok=True)
for item in ARTIFACTS:                                   # files or folders produced above
    dst = os.path.join(OUT, os.path.basename(item.rstrip("/")))
    (shutil.copytree(item, dst, dirs_exist_ok=True) if os.path.isdir(item) else shutil.copy2(item, dst))
def sha(p):
    h = hashlib.sha256(); h.update(open(p, "rb").read()); return h.hexdigest()
import sklearn, numpy
man_path = os.path.join(OUT, "manifest.json")
man = json.load(open(man_path)) if os.path.exists(man_path) else {"artifacts": {}}
for root, _, fs in os.walk(OUT):
    for f in fs:
        if f != "manifest.json":
            p = os.path.join(root, f); man["artifacts"][os.path.relpath(p, OUT)] = sha(p)
man.update({"exported_at": time.strftime("%Y-%m-%dT%H:%M:%S"), "python": platform.python_version(),
            "versions": {"scikit-learn": sklearn.__version__, "numpy": numpy.__version__}})
json.dump(man, open(man_path, "w"), indent=1)
shutil.make_archive("/content/drive/MyDrive/aeronexus/aeronexus_export", "zip", OUT)
print("saved to Drive: MyDrive/aeronexus/aeronexus_export.zip  →", sorted(man["artifacts"])[:12])

**Serve it (GPU box or cloud VM) and point the backend at it:**

```bash
vllm serve Qwen/Qwen2.5-VL-3B-Instruct --enable-lora --lora-modules drawings=./vlm-lora --max-model-len 8192
# backend
AERONEXUS_LLM_PROVIDER=openai OPENAI_BASE_URL=http://<host>:8000/v1 AERONEXUS_SMALL_MODEL=drawings \
AERONEXUS_LARGE_MODEL=Qwen/Qwen2.5-VL-3B-Instruct python -m aeronexus.evals.runner
```
The regression gate decides whether the fine-tune ships — not the training loss.